# 03 — Recurrence Quantification Analysis

**Purpose:** Compute RQA measures and autocorrelation statistics for every segmented trial.

Two analysis modes are supported:

- **Fixed-epsilon:** A single radius (in z-score units) is used for all trials.
- **RR-locked:** The radius is chosen per trial so that the recurrence rate matches a target value.

## Pipeline

1. Point `TRIALS_DIR` at the folder of segmented trial CSVs (output of Notebook 02).
2. Choose analysis mode and parameters.
3. Run the summary builder — it loops over all trials, z-scores the gaze angle, computes ACF and RQA.
4. Inspect and save the results.

## Outputs

- Summary CSV with one row per trial, including: filename metadata, mean/std gaze angle, ACF mean/AUC, and RQA measures (RR, DET, MaxL, ENT, L, LAM, TT).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from neon_gaze.rqa import build_rqa_summary
from neon_gaze.plotting import plot_recurrence_matrix

## Configuration

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Folder containing per-trial segmented CSVs
TRIALS_DIR = "../data/walk_segmented_csvs"

# Output directory for summary CSVs
OUTPUT_DIR = "../data/output"

# ── Analysis mode: choose ONE ──
# Option A: Fixed epsilon (radius in z-score units)
EPSILON = 0.07
TARGET_RR = None

# Option B: RR-locked (uncomment these and comment out Option A)
# EPSILON = None
# TARGET_RR = 0.05

# Minimum line length for DET, MaxL, ENT, LAM, TT
L_MIN = 5

# Set to True to save the summary CSV
SAVE_OUTPUT = False

## Compute RQA summary

In [ ]:
summary_df = build_rqa_summary(
    base_dir=TRIALS_DIR,
    epsilon=EPSILON,
    target_rr=TARGET_RR,
    l_min=L_MIN,
)

print(f"Processed {len(summary_df)} trials.")
display(summary_df.head(10))

In [ ]:
display(summary_df.describe())

## Save summary

In [ ]:
if SAVE_OUTPUT:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if TARGET_RR is not None:
        fname = f"rqa_summary_rr-locked-{TARGET_RR}_lmin-{L_MIN}.csv"
    else:
        fname = f"rqa_summary_eps-{EPSILON}_lmin-{L_MIN}.csv"
    out_path = os.path.join(OUTPUT_DIR, fname)
    summary_df.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")
else:
    print("SAVE_OUTPUT is False — set to True in the config cell to save.")

## Recurrence plot viewer

Use the widgets below to pick a trial CSV and adjust epsilon, then view the recurrence plot.

In [ ]:
import numpy as np
import pandas as pd
from ipywidgets import Dropdown, FloatSlider, HBox, VBox
from IPython.display import display, clear_output

csv_files = sorted(
    f for f in os.listdir(TRIALS_DIR) if f.lower().endswith(".csv")
) if os.path.isdir(TRIALS_DIR) else []

if csv_files:
    file_dropdown = Dropdown(
        options=csv_files,
        value=csv_files[0],
        description="Segment:",
        style={"description_width": "initial"},
        layout={"width": "400px"},
    )
    epsilon_slider = FloatSlider(
        value=0.07, min=0.02, max=0.50, step=0.01,
        description="epsilon (z):",
        style={"description_width": "initial"},
        readout_format=".2f",
        layout={"width": "400px"},
    )
    controls = HBox([file_dropdown, epsilon_slider])

    def update_plot(*_):
        fname = file_dropdown.value
        eps = float(epsilon_slider.value)
        clear_output(wait=True)
        display(VBox([controls]))
        df = pd.read_csv(os.path.join(TRIALS_DIR, fname))
        gaze = df["gaze angle [deg]"].to_numpy(dtype=float)
        plot_recurrence_matrix(gaze, epsilon=eps, title=f"Recurrence — {fname}")

    file_dropdown.observe(update_plot, names="value")
    epsilon_slider.observe(update_plot, names="value")
    display(VBox([controls]))
    update_plot()
else:
    print(f"No trial CSVs found in '{TRIALS_DIR}'.")